In [5]:
import os
import json
from openai import OpenAI


PROMPT = """
你的角色：你是一位经验丰富的电商广告文案专家，尤其擅长撰写生动、吸引人且能促进销售的服装类商品描述。

任务：请根据用户提供的【关键词列表】，为一件商品创作一篇完整的广告文案。

背景信息与输入：
关键词：用户将提供一组由“属性#值”构成的关键词，详细描述了商品的类型、版型、风格、图案、款式等特征。

示例参考：
输入关键词：类型#上衣*版型#宽松*风格#街头*风格#休闲*风格#朋克*图案#字母*图案#文字*图案#印花*衣样式#卫衣*衣款式#连帽*衣款式#对称
输出文案示例：个性休闲风的连帽卫衣造型时髦大方，宽松的版型剪裁让肉肉的小宝贝也可以穿着，保暖的连帽设计时刻给予宝贝温柔的呵护，袖子和后背别致时髦的字母印花点缀，满满的街头元素融入，演绎休闲朋克风，对称的小口袋美观大方，方便放置更多的随身物品。

文案创作要求：
完整性：生成的文案必须是一段通顺、完整、可直接使用的商品描述段落。
信息整合：需要自然地涵盖并突出关键词中提供的所有核心属性（如版型、风格、图案、款式等），不能遗漏关键词和卖点。
语言风格：语言应 亲切、有感染力、具有销售力。适当使用引导语，描绘穿着场景和感受，激发购买欲望。
目标导向：文案的目的是促进销售，因此要突出商品的优势、解决用户的痛点（如“宽松版型”修饰身材）、并描绘穿着后的美好体验。

输出格式：
记住，只输出最终的广告文案，无需额外解释或输出其他内容。

关键词：{}
输出："""

llm_client = OpenAI(
    api_key="EMPTY",
    base_url="http://localhost:8000/v1"
)

def request_chat(query, answer=""):
    try:
        prompt = PROMPT.format(query)
    
        completion = llm_client.chat.completions.create(
            model="phi-4-mini",
            messages=[
                {"role": "user", "content": prompt}
            ],
            max_tokens=2048,
            temperature=0.6
        )
        result = completion.choices[0].message.content
        return query, answer, result
        
    except:
       return query, "", ""

In [6]:
# 测试
query = "类型#上衣*版型#宽松*风格#日系*风格#简约*衣样式#卫衣"
_, _, res = request_chat(query)
print(res)

日系简约风的宽松卫衣，完美的搭配，是日常休闲场合的完美选择。舒适的宽松版型，赋予你无尽的自由穿着感，轻松自在地穿在你的日常生活中。简约的日系风格，简洁而不失品味，彰显着你的品味。无缝贴合的设计，带来完美的贴身感，追求完美的舒适与美观。无论是晨跑还是午睡，这款卫衣都能成为你不可或缺的伴侣。让简约风格与舒适结合，体验日系简约风的魅力。


In [7]:
train_data = [json.loads(line) for line in open("./data/train_dpo.json")]
test_data = [json.loads(line) for line in open("./data/test_dpo.json")]

In [8]:
print("train size:", len(train_data), "test size:", len(test_data))

train size: 45291 test size: 1070


In [9]:
import json
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed 

max_workers = 200
fw = open("data/train_dpo_final.json", "w")

# 使用多进程并行调用生成答案
with ProcessPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(request_chat, item["prompt"], item["chosen"]) for item in train_data]
    for future in tqdm(as_completed(futures), total=len(train_data), desc="Generations"):
        query, chosen_answer, rejected_answer = future.result()
        if rejected_answer and len(rejected_answer) < 500:
            info = {"prompt": query, "chosen": chosen_answer, "rejected": rejected_answer}
            fw.write(json.dumps(info, ensure_ascii=False) + "\n")
fw.close()

Generations: 100%|██████████| 45291/45291 [44:47<00:00, 16.85it/s]    


In [10]:
fw = open("data/test_dpo_final.json", "w")

# 使用多进程并行调用生成答案
with ProcessPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(request_chat, item["prompt"], item["chosen"]) for item in test_data]
    for future in tqdm(as_completed(futures), total=len(test_data), desc="Generations"):
        query, chosen_answer, rejected_answer = future.result()
        if rejected_answer and len(rejected_answer) < 500:
            info = {"prompt": query, "chosen": chosen_answer, "rejected": rejected_answer}
            fw.write(json.dumps(info, ensure_ascii=False) + "\n")
fw.close()

Generations: 100%|██████████| 1070/1070 [01:07<00:00, 15.95it/s]
